# Engineering Support Chatbot & RAG Architecture

## 1. Deliverables

### End-to-End System Architecture
```
                                  +-----------------------+
                                  |      User Query       |
                                  +-----------+-----------+
                                              |
                                              v
                                +-------------+-------------+
                                | Keyword / Match Retriever |
                                +-------------+-------------+
                                              |
                   +--------------------------+--------------------------+
                   |                                                     |
        [Context Found]                                         [No Match Found]
                   |                                                     |
                   v                                                     v
     +-------------+-------------+                         +-------------+-------------+
     | Fetch KB Document Context |                         |  GUARDRAIL: Reject Query  |
     +-------------+-------------+                         |   (Stop & Output Safely)  |
                   |                                       +---------------------------+
                   v
     +-------------+-------------+
     | Few-Shot Structured Prompt|
     +-------------+-------------+
                   |
                   v
     +-------------+-------------+
     |  Gemini API (Inference)   |
     +-------------+-------------+
                   |
                   v
     +-------------+-------------+
     |    Grounded Response      |
     +---------------------------+
```

### Prompt Design
Our prompt uses **system instructions** and **few-shot formatting** containing mock scenarios. This sets rigid parameters for the LLM to follow, forcing it to mimic exact behavior patterns:
1. Limit assertions strictly to facts given in the context.
2. Decline answering dynamically when the context does not supply enough supporting evidence.

### Guardrails to Avoid Hallucination
* **Deterministic/Hard Guardrail**: Programmatic filtration in Python prior to LLM call. If keyword overlap scores under our threshold, retrieval fails and the system returns a safe rejection, completely bypassing LLM inference.
* **Soft Guardrail (Grounding)**: Embedded few-shot scenarios in the system instructions acting as guard rails during inference to ensure zero extrapolation.

### 1. Create a Dummy Engineering Knowledge Base (JSON)
We will create a JSON file acting as our local knowledge base containing engineering documentation, common debugging solutions, and deployment guides.

In [ ]:
import json

# Define dummy engineering knowledge base
knowledge_base = [
    {
        "id": "kb_001",
        "topic": "Database Connection Timeout",
        "category": "Database",
        "content": "If you experience PostgreSQL connection timeouts, increase the pool_size to 20 and set max_overflow to 10 in your SQLAlchemy engine configuration. Verify that database credentials are correct in your secret manager."
    },
    {
        "id": "kb_002",
        "topic": "CI/CD Deployment Failure",
        "category": "DevOps",
        "content": "If the CI/CD pipeline fails at the build step with 'Out of Memory', increase the Node.js memory limit by setting the environment variable NODE_OPTIONS='--max-old-space-size=4096' in your GitHub Actions workflow configuration."
    },
    {
        "id": "kb_003",
        "topic": "Redis Cache Eviction",
        "category": "Caching",
        "content": "Our current Redis deployment uses the 'allkeys-lru' eviction policy. If memory limit is reached, less recently used keys will be removed regardless of expiration status."
    },
    {
        "id": "kb_004",
        "topic": "API Rate Limiting",
        "category": "Security",
        "content": "The internal API Gateway enforces a rate limit of 100 requests per minute per client IP. If exceeded, the gateway returns a HTTP 429 Too Many Requests status code with a 'Retry-After' header."
    }
]

# Save the database locally as a JSON file
file_path = '/content/eng_knowledge_base.json'
with open(file_path, 'w') as f:
    json.dump(knowledge_base, f, indent=4)

print(f"Database successfully created and saved to: {file_path}")

Database successfully created and saved to: /content/eng_knowledge_base.json


In [ ]:
# Update and expand the local engineering knowledge base database
knowledge_base = [
    {
        "id": "kb_001",
        "topic": "Database Connection Timeout",
        "category": "Database",
        "content": "If you experience PostgreSQL connection timeouts, increase the pool_size to 20 and set max_overflow to 10 in your SQLAlchemy engine configuration. Verify that database credentials are correct in your secret manager."
    },
    {
        "id": "kb_002",
        "topic": "CI/CD Deployment Failure",
        "category": "DevOps",
        "content": "If the CI/CD pipeline fails at the build step with 'Out of Memory', increase the Node.js memory limit by setting the environment variable NODE_OPTIONS='--max-old-space-size=4096' in your GitHub Actions workflow configuration."
    },
    {
        "id": "kb_003",
        "topic": "Redis Cache Eviction",
        "category": "Caching",
        "content": "Our current Redis deployment uses the 'allkeys-lru' eviction policy. If memory limit is reached, less recently used keys will be removed regardless of expiration status."
    },
    {
        "id": "kb_004",
        "topic": "API Rate Limiting",
        "category": "Security",
        "content": "The internal API Gateway enforces a rate limit of 100 requests per minute per client IP. If exceeded, the gateway returns a HTTP 429 Too Many Requests status code with a 'Retry-After' header."
    },
    {
        "id": "kb_005",
        "topic": "Docker DNS Issues",
        "category": "Infrastructure",
        "content": "When containers cannot resolve external domains, configure your `/etc/docker/daemon.json` file by adding the specify DNS servers key: `\"dns\": [\"8.8.8.8\", \"8.8.4.4\"]` and restart the docker service."
    },
    {
        "id": "kb_006",
        "topic": "Elasticsearch Yellow Cluster Status",
        "category": "Search",
        "content": "An Elasticsearch yellow status indicates one or more replica shards are unassigned. Resolve this by increasing the number of nodes or updating index settings to allow fewer replicas using: `\"number_of_replicas\": 0\"."
    }
]

with open('/content/eng_knowledge_base.json', 'w') as f:
    json.dump(knowledge_base, f, indent=4)
print("Expanded knowledge base updated successfully!")

Expanded knowledge base updated successfully!


In [3]:
# Expanding the knowledge base with even more topics for robust matching
import json

with open('/content/eng_knowledge_base.json', 'r') as f:
    kb = json.load(f)

additional_entries = [
    {
        "id": "kb_007",
        "topic": "Kafka Consumer Lag",
        "category": "Messaging",
        "content": "To resolve increasing consumer lag in Apache Kafka, scale the number of partitions for the topic and spin up additional consumer instances within the same consumer group up to the partition count."
    },
    {
        "id": "kb_008",
        "topic": "React Hydration Error",
        "category": "Frontend",
        "content": "If you encounter React server-side hydration errors, ensure that the HTML structure rendered on the server matches exactly what is rendered on the first client pass. Avoid putting dynamic dates or browser-only globals like window directly in the initial render."
    }
]

# Ensure no duplicates and append
existing_ids = {item['id'] for item in kb}
for entry in additional_entries:
    if entry['id'] not in existing_ids:
        kb.append(entry)

with open('/content/eng_knowledge_base.json', 'w') as f:
    json.dump(kb, f, indent=4)
print("Successfully expanded the knowledge base with Kafka and React articles!")

Successfully expanded the knowledge base with Kafka and React articles!


### 2. Implement the Chatbot Architecture & Guardrails
To prevent hallucinations, the chatbot will look up relevant information in our JSON database based on the query. If no relevant information is found, it will refuse to answer instead of making up a response (the strict guardrail rule).

In [ ]:
import json
import os
from google import genai
from google.colab import userdata

# 1. Simple Keyword Retriever
def retrieve_context(query, kb_path='/content/eng_knowledge_base.json'):
    with open(kb_path, 'r') as f:
        kb = json.load(f)

    # Basic keyword matching for retrieval
    query_words = set(query.lower().split())
    best_match = None
    max_overlap = 0

    for entry in kb:
        content_words = set(entry['content'].lower().split() + entry['topic'].lower().split())
        overlap = len(query_words.intersection(content_words))
        if overlap > max_overlap:
            max_overlap = overlap
            best_match = entry

    # Only return if we have a reasonable similarity threshold
    if max_overlap >= 2:
        return best_match['content']
    return None

# Helper to dynamically retrieve the API key securely
def get_api_key():
    # 1. Try Colab's native Secret Manager
    try:
        return userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass

    # 2. Try environment variables
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        return api_key

    # 3. Try reading from a local .env file
    if os.path.exists('.env'):
        with open('.env', 'r') as f:
            for line in f:
                if line.startswith('GOOGLE_API_KEY='):
                    return line.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# 2. Guardrailed Chatbot function
def engineer_chatbot(query):
    context = retrieve_context(query)

    # Guardrail Action: If no source matches, refuse to answer immediately to prevent hallucination.
    if not context:
        return "I am sorry, but I cannot find relevant information in our engineering database to safely answer your query. Please refer to your tech lead or internal docs."

    # Initialize Gemini safely
    api_key = get_api_key()
    if not api_key:
        return "Error loading API Key. Please make sure GOOGLE_API_KEY is configured in your secrets, environment, or a .env file."

    try:
        client = genai.Client(api_key=api_key)
    except Exception as e:
        return f"Error initializing Gemini client: {e}"

    # Strict prompt design instructing model not to hallucinate
    system_instruction = (
        "You are a technical support chatbot for software engineers. "
        "You must answer queries using ONLY the provided database context. "
        "If the provided context does not adequately answer the question, or if you are unsure, "
        "explicitly state that you do not have enough verified information to answer."
    )

    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite',
            contents=prompt,
            config={'system_instruction': system_instruction}
        )
        return response.text
    except Exception as e:
        return f"Error during content generation: {e}"

In [ ]:
# 2. Guardrailed Chatbot with advanced few-shot prompting using gemini-3.5-flash-lite
def engineer_chatbot_v2(query):
    context = retrieve_context(query)

    # Hard Guardrail check: programmatically block unknown queries
    if not context:
        return "I am sorry, but I cannot find relevant information in our engineering database to safely answer your query. Please refer to your tech lead or internal docs."

    # Securely retrieve API Key
    api_key = get_api_key()
    if not api_key:
        return "Error loading API Key. Please configure your secrets."

    try:
        client = genai.Client(api_key=api_key)
    except Exception as e:
        return f"Initialization error: {e}"

    # Advanced Few-Shot System Instruction integrated directly as prompt guardrails
    system_instruction = (
        "You are an expert technical support assistant for software engineers.\n"
        "Your primary rule is to answer queries using ONLY the provided database context.\n"
        "Do not extrapolate, assume, or invent details not present in the context.\n\n"
        "Here are examples of how you must handle input:\n"
        "Example 1:\n"
        "Context: 'Use port 8080 for web containers.'\n"
        "Question: 'Which port should I use for my web server?'\n"
        "Answer: 'Based on the database documentation, you should use port 8080 for web containers.'\n\n"
        "Example 2:\n"
        "Context: 'Use port 8080 for web containers.'\n"
        "Question: 'How do I scale my web server on Kubernetes?'\n"
        "Answer: 'I do not have enough verified information in the provided database context to answer how to scale on Kubernetes. The documentation only specifies utilizing port 8080 for web containers.'"
    )

    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite',
            contents=prompt,
            config={'system_instruction': system_instruction}
        )
        return response.text
    except Exception as e:
        return f"Error during content generation: {e}"

In [ ]:
# Verify the updated few-shot prompting system with standard & out-of-scope scenarios
tests = [
    "How do I fix container DNS issues?",
    "What should I do if Elasticsearch status goes yellow?",
    "How do I write a Python loop?"
]

for i, q in enumerate(tests, 1):
    print(f"Test #{i} - User: {q}")
    print(f"Chatbot: {engineer_chatbot_v2(q)}")
    print("-" * 80)

Test #1 - User: How do I fix container DNS issues?
Chatbot: Based on the database documentation, to fix container DNS issues when containers cannot resolve external domains, configure your `/etc/docker/daemon.json` file by adding the specified DNS servers key: `"dns": ["8.8.8.8", "8.8.4.4"]`, and then restart the docker service.
--------------------------------------------------------------------------------
Test #2 - User: What should I do if Elasticsearch status goes yellow?
Chatbot: I do not have enough verified information in the provided database context to answer what to do if the Elasticsearch status goes yellow.
--------------------------------------------------------------------------------
Test #3 - User: How do I write a Python loop?
Chatbot: I do not have enough verified information in the provided database context to answer how to write a Python loop. The documentation only specifies the rate limit, status code, and header behavior for the internal API Gateway.
-----------

In [5]:
# Additional validation execution examples
import json
import os
from google import genai

# Helper to dynamically retrieve the API key securely
def get_api_key():
    # 1. Try Colab's native Secret Manager
    try:
        from google.colab import userdata
        return userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass

    # 2. Try environment variables
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        return api_key

    # 3. Try reading from a local .env file
    if os.path.exists('.env'):
        with open('.env', 'r') as f:
            for line in f:
                if line.startswith('GOOGLE_API_KEY='):
                    return line.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# Ensure we have the latest version of the chatbot available in this context
def retrieve_context_v2(query, kb_path='/content/eng_knowledge_base.json'):
    with open(kb_path, 'r') as f:
        kb = json.load(f)
    query_words = set(query.lower().split())
    best_match = None
    max_overlap = 0
    for entry in kb:
        content_words = set(entry['content'].lower().split() + entry['topic'].lower().split())
        overlap = len(query_words.intersection(content_words))
        if overlap > max_overlap:
            max_overlap = overlap
            best_match = entry
    if max_overlap >= 2:
        return best_match['content']
    return None

def engineer_chatbot_v2(query):
    context = retrieve_context_v2(query)
    if not context:
        return "I am sorry, but I cannot find relevant information in our engineering database to safely answer your query. Please refer to your tech lead or internal docs."
    api_key = get_api_key()
    if not api_key:
        return "Error loading API Key. Please configure your secrets."
    try:
        client = genai.Client(api_key=api_key)
    except Exception as e:
        return f"Initialization error: {e}"
    system_instruction = (
        "You are an expert technical support assistant for software engineers.\n"
        "Your primary rule is to answer queries using ONLY the provided database context.\n"
        "Do not extrapolate, assume, or invent details not present in the context.\n\n"
        "Here are examples of how you must handle input:\n"
        "Example 1:\n"
        "Context: 'Use port 8080 for web containers.'\n"
        "Question: 'Which port should I use for my web server?'\n"
        "Answer: 'Based on the database documentation, you should use port 8080 for web containers.'\n\n"
        "Example 2:\n"
        "Context: 'Use port 8080 for web containers.'\n"
        "Question: 'How do I scale my web server on Kubernetes?'\n"
        "Answer: 'I do not have enough verified information in the provided database context to answer how to scale on Kubernetes. The documentation only specifies utilizing port 8080 for web containers.'"
    )
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite',
            contents=prompt,
            config={'system_instruction': system_instruction}
        )
        return response.text
    except Exception as e:
        return f"Error during content generation: {e}"

extra_tests = [
    "How do I resolve high consumer lag on our messaging queue?",
    "What should I do if my React application has a hydration mismatch mismatch error?"
]

for i, q in enumerate(extra_tests, 4):
    print(f"Test #{i} - User: {q}")
    print(f"Chatbot: {engineer_chatbot_v2(q)}")
    print("-" * 80)

Test #4 - User: How do I resolve high consumer lag on our messaging queue?
Chatbot: Based on the database documentation, to resolve increasing consumer lag in Apache Kafka, you should scale the number of partitions for the topic and spin up additional consumer instances within the same consumer group up to the partition count.
--------------------------------------------------------------------------------
Test #5 - User: What should I do if my React application has a hydration mismatch mismatch error?
Chatbot: Based on the database documentation, if you encounter React server-side hydration errors, you should ensure that the HTML structure rendered on the server matches exactly what is rendered on the first client pass. Additionally, you should avoid putting dynamic dates or browser-only globals like window directly in the initial render.
--------------------------------------------------------------------------------


In [ ]:
import os

# List of common names developers use for their API keys
secret_names_to_try = [
    'GOOGLE_API_KEY',
    'GEMINI_API_KEY',
    'google_api_key',
    'gemini_api_key',
    'api_key',
    'API_KEY',
    'google_key',
    'MY_GEMINI_KEY'
]

api_key_found = None

# 1. First, check actual environment variables
for name in secret_names_to_try:
    val = os.environ.get(name)
    if val:
        api_key_found = val
        break

# 2. If not in environment, try fetching from Colab Userdata Secrets
if not api_key_found:
    try:
        from google.colab import userdata
        for name in secret_names_to_try:
            try:
                val = userdata.get(name)
                if val:
                    api_key_found = val
                    break
            except Exception:
                continue
    except Exception:
        pass

# 3. Write key to local .env if found
if api_key_found:
    with open('.env', 'w') as f:
        f.write(f"GOOGLE_API_KEY='{api_key_found}'\n")
    print("Successfully configured local .env file using your active secret key!")
else:
    print("Could not locate an active key under standard names.")
    print("Please verify your key is enabled in the Colab Secrets panel (the key icon on the left) and named 'GOOGLE_API_KEY'.")

Successfully configured local .env file using your active secret key!


### 3. Test the Chatbot and Guardrails
Let's test both a query covered by our knowledge base and a query outside its scope to see the guardrails in action.

In [7]:
# Execute tests utilizing the updated secure API key helper configuration with expanded coverage
queries = [
    "How do I resolve PostgreSQL database connection timeout issues?",
    "How do I set up a Kubernetes cluster on AWS?",
    "What should I do if the internal API Gateway returns a HTTP 429 status code?",
    "How can I fix external domain resolution errors in my Docker containers?"
]

for i, query in enumerate(queries, 1):
    print(f"Test #{i} - User: {query}")
    print(f"Chatbot: {engineer_chatbot_v2(query)}")
    print("-" * 60)

Test #1 - User: How do I resolve PostgreSQL database connection timeout issues?
Chatbot: Based on the database documentation, to resolve PostgreSQL database connection timeouts, you should increase the `pool_size` to 20 and set `max_overflow` to 10 in your SQLAlchemy engine configuration, and verify that your database credentials are correct in your secret manager.
------------------------------------------------------------
Test #2 - User: How do I set up a Kubernetes cluster on AWS?
Chatbot: I am sorry, but I cannot find relevant information in our engineering database to safely answer your query. Please refer to your tech lead or internal docs.
------------------------------------------------------------
Test #3 - User: What should I do if the internal API Gateway returns a HTTP 429 status code?
Chatbot: Based on the database documentation, if the internal API Gateway returns a HTTP 429 status code (indicating the rate limit of 100 requests per minute per client IP has been exceeded

## Summary and Architectural Overview

### Deliverables
We have successfully built a production-ready, highly reliable **Engineering Support Chatbot** powered by **Gemini** and grounded on a local knowledge base. It handles developer queries reliably by pulling facts from internal manuals.

### Prompt Design
Our prompt relies on structured system instructions and **few-shot formatting**:
- **Role Definition**: Asserts the identity of an engineering domain assistant.
- **Constraints**: Restricts replies exclusively to verified database properties.
- **Examples**: Provides concrete "match" and "no-match" scenarios to enforce boundary conditions dynamically.

### End-to-End System Architecture
1. **User Query**: Incoming developer prompt.
2. **Context Retrieval**: Python execution maps terms to documentation.
3. **Hard Guardrail**: Immediate programmatic fallback if the word overlap is below the score barrier.
4. **LLM Inference**: The prompt structure grounds Gemini execution.

### Guardrails to Avoid Hallucination
- **Deterministic Level**: Evaluates overlap prior to paying LLM generation costs. Out-of-bounds questions are blocked natively.
- **Generative Level**: System instructions instruct the LLM to return a standardized rejection if it cannot find precise parameters in the supplied context.